In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ Step 1/5 — Imports done")

# ─────────────────────────────────────────────
# IMAGES
# ─────────────────────────────────────────────
def make_fake_satellite(day, size=64):
    np.random.seed(int(day * 100))
    img = np.zeros((size, size, 3), dtype=np.float32)
    img[:,:,0] = 0.55 + np.random.normal(0, 0.03, (size,size))
    img[:,:,1] = 0.45 + np.random.normal(0, 0.03, (size,size))
    img[:,:,2] = 0.30 + np.random.normal(0, 0.03, (size,size))
    veg = 0.15 + day * 0.25
    img[:size//2,:size//2,1] += veg
    img[:size//2,:size//2,0] -= veg * 0.4
    img[size//2:,size//2:,2] = 0.55
    img[size//2:,size//2:,0] = 0.10
    img[size//2:,size//2:,1] = 0.20
    return np.clip(img, 0, 1)

days   = [i/10.0 for i in range(11)]
images = [make_fake_satellite(d) for d in days]
print(f"✅ Step 2/5 — {len(images)} images ready")

# ─────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────
class SatelliteDataset(Dataset):
    def __init__(self, images):
        self.samples = [(images[i], images[i+1], images[i+2])
                        for i in range(len(images)-2)]
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        b, m, a = self.samples[idx]
        return (torch.tensor(b).permute(2,0,1),
                torch.tensor(a).permute(2,0,1),
                torch.tensor(m).permute(2,0,1))

dataloader = DataLoader(SatelliteDataset(images), batch_size=2, shuffle=True)
print(f"✅ Step 3/5 — Dataset ready")

# ─────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────
class TempoSatModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(
            nn.Conv2d(6, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU())
        self.enc2 = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU())
        self.up1  = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec1 = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU())
        self.out  = nn.Sequential(nn.Conv2d(16, 3, 1), nn.Sigmoid())

    def forward(self, b, a):
        x  = torch.cat([b, a], dim=1)
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        d1 = self.up1(e2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return self.out(d1)

model = TempoSatModel()
print(f"✅ Step 4/5 — Model built")

# ─────────────────────────────────────────────
# TRAINING
# ─────────────────────────────────────────────
optimizer = optim.Adam(model.parameters(), lr=0.0005)  # slower, more careful learning

# ── Smarter loss: MSE + SSIM combined ─────────────────────────────────────
def ssim_loss(pred, target):
    """
    SSIM measures structural similarity — much smarter than raw pixel error.
    Looks at contrast, brightness and structure together like a human would.
    """
    mu_p    = pred.mean()
    mu_t    = target.mean()
    sig_p   = pred.var()
    sig_t   = target.var()
    sig_pt  = ((pred - mu_p) * (target - mu_t)).mean()
    c1, c2  = 0.01**2, 0.03**2
    ssim    = ((2*mu_p*mu_t + c1) * (2*sig_pt + c2)) / \
              ((mu_p**2 + mu_t**2 + c1) * (sig_p + sig_t + c2))
    return 1 - ssim   # we want to minimise, so flip it

def combined_loss(pred, target):
    mse  = nn.MSELoss()(pred, target)
    ssim = ssim_loss(pred, target)
    return 0.5 * mse + 0.5 * ssim   # 50/50 blend of both

# ── Training with smarter loss ─────────────────────────────────────────────
best_loss = float('inf')
print("🚀 Training with SSIM + MSE combined loss...\n")

for epoch in range(80):
    epoch_loss = 0.0
    for b, a, t in dataloader:
        optimizer.zero_grad()
        pred = model(b, a)
        loss = combined_loss(pred, t)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg = epoch_loss / len(dataloader)
    if avg < best_loss:
        best_loss = avg

    if (epoch+1) % 10 == 0:
        print(f"   Epoch {epoch+1:2d}/80 — Loss: {avg:.6f}  Best so far: {best_loss:.6f}")

print(f"\n✅ Training complete! Best loss achieved: {best_loss:.6f}")

# ── Save results ───────────────────────────────────────────────────────────
import cv2

model.eval()
day1  = make_fake_satellite(0.0)
day5  = make_fake_satellite(0.5)
day10 = make_fake_satellite(1.0)
base  = (day1 + day10) / 2.0

with torch.no_grad():
    t1   = torch.tensor(day1).permute(2,0,1).unsqueeze(0)
    t10  = torch.tensor(day10).permute(2,0,1).unsqueeze(0)
    pred = model(t1, t10).squeeze(0).permute(1,2,0).numpy()

def psnr(p, r):
    mse = np.mean((p - r)**2)
    return 10 * np.log10(1.0/mse) if mse > 0 else 999.0

def save_image_cv2(arr, path):
    img_uint8 = (arr * 255).astype(np.uint8)
    img_bgr   = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR)
    img_big   = cv2.resize(img_bgr, (320, 320), interpolation=cv2.INTER_NEAREST)
    cv2.imwrite(path, img_big)

os.makedirs("../data/results", exist_ok=True)
save_image_cv2(day1, "../data/results/1_day1_input.png")
save_image_cv2(base, "../data/results/2_baseline_prediction.png")
save_image_cv2(pred, "../data/results/3_AI_prediction.png")
save_image_cv2(day5, "../data/results/4_real_day5.png")

b_score  = psnr(base, day5)
ai_score = psnr(pred,  day5)

print(f"\n📊 Final Results:")
print(f"   Baseline PSNR : {b_score:.2f} dB")
print(f"   AI model PSNR : {ai_score:.2f} dB")
print(f"   Improvement   : +{ai_score - b_score:.2f} dB")
print(f"\n   {'✅ AI model WON!' if ai_score > b_score else '🔄 Getting closer...'}")
print(f"\n📁 Images saved to: Documents → TempoSat_Project → data → results")

✅ Step 1/5 — Imports done
✅ Step 2/5 — 11 images ready
✅ Step 3/5 — Dataset ready
✅ Step 4/5 — Model built
🚀 Training with SSIM + MSE combined loss...

   Epoch 10/80 — Loss: 0.201728  Best so far: 0.201728
   Epoch 20/80 — Loss: 0.037379  Best so far: 0.037379
   Epoch 30/80 — Loss: 0.014853  Best so far: 0.014853
   Epoch 40/80 — Loss: 0.012576  Best so far: 0.012576
   Epoch 50/80 — Loss: 0.010976  Best so far: 0.010976
   Epoch 60/80 — Loss: 0.009621  Best so far: 0.009621
   Epoch 70/80 — Loss: 0.008666  Best so far: 0.008666
   Epoch 80/80 — Loss: 0.007985  Best so far: 0.007985

✅ Training complete! Best loss achieved: 0.007985

📊 Final Results:
   Baseline PSNR : 29.85 dB
   AI model PSNR : 30.55 dB
   Improvement   : +0.70 dB

   ✅ AI model WON!

📁 Images saved to: Documents → TempoSat_Project → data → results
